In [46]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/nfl.db")
df_full = pd.read_sql_query("SELECT * FROM games_with_features", conn)
conn.close()

print(f"Loaded {len(df_full)} games, {len(df_full.columns)} columns")

Loaded 2458 games, 95 columns


In [55]:
feature_cols_v2 = [
    'home_recent_form', 'away_recent_form',
    'home_recent_point_diff', 'away_recent_point_diff',
    'home_qb_recent_yards', 'away_qb_recent_yards',
    'home_qb_recent_tds', 'away_qb_recent_tds',
    'home_qb_recent_ints', 'away_qb_recent_ints',
    'home_qb_recent_epa', 'away_qb_recent_epa',
    'home_rb_recent_rush_yards', 'away_rb_recent_rush_yards',
    'home_rb_recent_rush_epa', 'away_rb_recent_rush_epa',
    'home_rb_recent_rec_yards', 'away_rb_recent_rec_yards',
    'home_wrte_recent_rec_yards', 'away_wrte_recent_rec_yards',
    'home_wrte_recent_rec_epa', 'away_wrte_recent_rec_epa',
    'home_wrte_recent_targets', 'away_wrte_recent_targets',
    'home_qb_injury_flag', 'away_qb_injury_flag',
    'home_rb_injury_flag', 'away_rb_injury_flag',
    'home_wrte_injury_flag', 'away_wrte_injury_flag',
    'home_epa_allowed_recent', 'away_epa_allowed_recent',
    'home_yards_allowed_recent', 'away_yards_allowed_recent',
    'home_takeaways_recent', 'away_takeaways_recent',
    'home_coach_h2h_wins', 'h2h_games_played',  # per Option 1: drop coach recent form, keep h2h
    'div_game'
]

X2 = df_full[feature_cols_v2].copy()
X2_imputed = pd.DataFrame(imputer.fit_transform(X2), columns=feature_cols_v2, index=X2.index)

scores_v2 = cross_val_score(model, X2_imputed, y, cv=tscv, scoring='accuracy')
print(f"Logistic Regression (EPA/yards allowed + takeaways, no team-coach-form redundancy): {scores_v2.mean():.3f} (+/- {scores_v2.std():.3f})")
print(f"Fold scores: {[round(s, 3) for s in scores_v2]}")

Logistic Regression (EPA/yards allowed + takeaways, no team-coach-form redundancy): 0.591 (+/- 0.018)
Fold scores: [np.float64(0.599), np.float64(0.599), np.float64(0.589), np.float64(0.557), np.float64(0.609)]


In [51]:
team_stats = nfl.load_team_stats(seasons=list(range(2015, 2024))).to_pandas()
team_stats = team_stats[['game_id', 'season', 'week', 'team', 'opponent_team',
                          'passing_epa', 'rushing_epa', 'passing_yards', 'rushing_yards',
                          'def_interceptions', 'fumble_recovery_opp']]

# Self-join: for each team's row, find their opponent's own offensive output in the same game
opponent_offense = team_stats[['game_id', 'team', 'passing_epa', 'rushing_epa', 'passing_yards', 'rushing_yards']].rename(
    columns={'team': 'opponent_team', 'passing_epa': 'opp_passing_epa', 'rushing_epa': 'opp_rushing_epa',
             'passing_yards': 'opp_passing_yards', 'rushing_yards': 'opp_rushing_yards'}
)

team_stats = team_stats.merge(opponent_offense, on=['game_id', 'opponent_team'], how='left')

# What this team's DEFENSE allowed = what their opponent's OFFENSE produced
team_stats['epa_allowed'] = team_stats['opp_passing_epa'] + team_stats['opp_rushing_epa']
team_stats['yards_allowed'] = team_stats['opp_passing_yards'] + team_stats['opp_rushing_yards']
team_stats['takeaways'] = team_stats['def_interceptions'] + team_stats['fumble_recovery_opp']

team_stats[['game_id', 'team', 'opponent_team', 'epa_allowed', 'yards_allowed', 'takeaways']].head(10)

,game_id,team,opponent_team,epa_allowed,yards_allowed,takeaways
0,2015_01_NO_ARI,ARI,NO,-1.920138,409,1
1,2015_01_PHI_ATL,ATL,PHI,5.400452,399,2
2,2015_01_BAL_DEN,BAL,DEN,-16.876773,244,1
3,2015_01_IND_BUF,BUF,IND,4.044229,307,3
4,2015_01_CAR_JAX,CAR,JAX,-16.419194,279,3
5,2015_01_GB_CHI,CHI,GB,18.195264,322,0
6,2015_01_CIN_OAK,CIN,LV,-11.012136,266,2
7,2015_01_CLE_NYJ,CLE,NYJ,4.703464,333,1
8,2015_01_NYG_DAL,DAL,NYG,2.859031,288,0
9,2015_01_BAL_DEN,DEN,BAL,-21.940909,190,2


In [52]:
# Team_stats already appears to use "current" codes - check what team codes actually exist here
print(sorted(team_stats['team'].unique()))

['ARI', 'ATL', 'BAL', 'BUF', 'CAR', 'CHI', 'CIN', 'CLE', 'DAL', 'DEN', 'DET', 'GB', 'HOU', 'IND', 'JAX', 'KC', 'LA', 'LAC', 'LV', 'MIA', 'MIN', 'NE', 'NO', 'NYG', 'NYJ', 'PHI', 'PIT', 'SEA', 'SF', 'TB', 'TEN', 'WAS']


In [53]:
team_stats = team_stats.sort_values(['team', 'season', 'week']).reset_index(drop=True)

for col in ['epa_allowed', 'yards_allowed', 'takeaways']:
    team_stats[f'recent_{col}'] = (
        team_stats.groupby(['team', 'season'])[col]
        .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
    )

team_stats[['team', 'season', 'week', 'epa_allowed', 'recent_epa_allowed']].head(10)

,team,season,week,epa_allowed,recent_epa_allowed
0,ARI,2015,1,-1.920138,NaN
1,ARI,2015,2,-13.344114,-1.920138
2,ARI,2015,3,-27.287458,-7.632126
3,ARI,2015,4,3.060655,-14.183903
4,ARI,2015,5,-18.059609,-9.872764
5,ARI,2015,6,0.494598,-11.510133
6,ARI,2015,7,2.421044,-11.027186
7,ARI,2015,8,-8.499176,-7.874154
8,ARI,2015,10,-1.600425,-4.116498
9,ARI,2015,11,5.438162,-5.048714


In [54]:
def_allowed_recent = team_stats[['team', 'game_id', 'recent_epa_allowed', 'recent_yards_allowed', 'recent_takeaways']]

home_def2 = def_allowed_recent.rename(columns={
    'team': 'home_team_std',
    'recent_epa_allowed': 'home_epa_allowed_recent',
    'recent_yards_allowed': 'home_yards_allowed_recent',
    'recent_takeaways': 'home_takeaways_recent'
})

away_def2 = def_allowed_recent.rename(columns={
    'team': 'away_team_std',
    'recent_epa_allowed': 'away_epa_allowed_recent',
    'recent_yards_allowed': 'away_yards_allowed_recent',
    'recent_takeaways': 'away_takeaways_recent'
})

df_full = df_full.drop(columns=[c for c in df_full.columns if 'epa_allowed_recent' in c or 'yards_allowed_recent' in c or 'takeaways_recent' in c], errors='ignore')

df_full = df_full.merge(home_def2, on=['home_team_std', 'game_id'], how='left')
df_full = df_full.merge(away_def2, on=['away_team_std', 'game_id'], how='left')

df_full[df_full['game_id'].isin(['2019_05_CHI_OAK', '2016_03_SD_IND'])][
    ['game_id', 'home_team', 'away_team', 'home_epa_allowed_recent', 'away_epa_allowed_recent']]

,game_id,home_team,away_team,home_epa_allowed_recent,away_epa_allowed_recent
310,2016_03_SD_IND,IND,SD,12.910116,1.528741
1137,2019_05_CHI_OAK,OAK,CHI,8.072278,-10.504904


In [56]:
def moneyline_to_prob(ml):
    if pd.isna(ml):
        return None
    if ml < 0:
        return -ml / (-ml + 100)
    else:
        return 100 / (ml + 100)

df_full['vegas_home_prob'] = df_full['home_moneyline'].apply(moneyline_to_prob)

df_full[['game_id', 'home_moneyline', 'away_moneyline', 'vegas_home_prob']].sample(10)

,game_id,home_moneyline,away_moneyline,vegas_home_prob
774,2017_17_GB_DET,-315.0,277.0,0.759036
844,2018_03_LAC_LA,-288.0,254.0,0.742268
693,2017_11_ATL_SEA,-114.0,103.0,0.532710
2252,2023_06_BAL_TEN,205.0,-250.0,0.327869
248,2015_17_NE_MIA,330.0,-380.0,0.232558
1338,2020_01_NYJ_BUF,-279.0,242.0,0.736148
937,2018_10_NO_CIN,225.0,-253.0,0.307692
1593,2020_18_TB_WAS,373.0,-456.0,0.211416
1290,2019_15_BUF_PIT,-112.0,101.0,0.528302
73,2015_05_NE_DAL,305.0,-350.0,0.246914


In [57]:
test_df = df_full[df_full['season'] >= 2022].copy()

# Vegas "prediction": home team favored if their implied probability > 50%
test_df['vegas_pred_home_win'] = (test_df['vegas_home_prob'] > 0.5).astype(int)
vegas_accuracy = (test_df['vegas_pred_home_win'] == test_df['home_win']).mean()

print(f"Vegas accuracy (2022-2023): {vegas_accuracy:.3f}")
print(f"Your model accuracy (2022-2023, from earlier CV): ~0.591")

Vegas accuracy (2022-2023): 0.663
Your model accuracy (2022-2023, from earlier CV): ~0.591


In [58]:
train_mask = df_full['season'] <= 2021
test_mask = df_full['season'] >= 2022

X2_train, X2_test = X2_imputed[train_mask], X2_imputed[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

model.fit(X2_train, y_train)
model_test_accuracy = model.score(X2_test, y_test)

print(f"Your model accuracy (2022-2023 specifically): {model_test_accuracy:.3f}")
print(f"Vegas accuracy (2022-2023): {vegas_accuracy:.3f}")


Your model accuracy (2022-2023 specifically): 0.601
Vegas accuracy (2022-2023): 0.663


In [59]:
test_df['model_prob'] = model.predict_proba(X2_test)[:, 1]
test_df['model_pred'] = (test_df['model_prob'] > 0.5).astype(int)

disagreement = (test_df['model_pred'] != test_df['vegas_pred_home_win'])
print(f"Model and Vegas disagree on {disagreement.sum()} of {len(test_df)} games ({disagreement.mean():.1%})")

# When they disagree, who's actually right more often?
disagree_df = test_df[disagreement]
model_right_when_disagree = (disagree_df['model_pred'] == disagree_df['home_win']).mean()
vegas_right_when_disagree = (disagree_df['vegas_pred_home_win'] == disagree_df['home_win']).mean()

print(f"When they disagree: model correct {model_right_when_disagree:.1%}, Vegas correct {vegas_right_when_disagree:.1%}")

Model and Vegas disagree on 135 of 569 games (23.7%)
When they disagree: model correct 37.0%, Vegas correct 63.0%
